In [0]:
from pyspark.sql.functions import sum, current_timestamp, col

gold_schema = dbutils.widgets.get("gold_schema")
silver_schema = dbutils.widgets.get("silver_schema")

# step 1: ensure gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# step 2: load the silver tables
orders = spark.table(f"{silver_schema}.orders_dedupe")
customers = spark.table(f"{silver_schema}.customers_enriched")

# step 3: Join & aggregate by city
city_wise_sales = (
orders.join(customers, "customer_id")
.filter(col("city").isNotNull() & col("state").isNotNull())
.groupBy("state", "city")
.agg(sum("total_amount").alias("total_sales"))
)

# step 4: write it to the gold table
city_wise_sales.write.mode("overwrite").saveAsTable(f"{gold_schema}.city_wise_sales")

print("gold layer created")



gold layer created
